# F1 Pit Stop — LSTM (PyTorch / CUDA 버전)

기존 Keras 노트북을 **PyTorch로 포팅**하여 TabTransformer와 동일한 cu128(CUDA) 환경에서 실행한다.
구조·설계는 `C289039_유승환_LSTM_실험계획.md` 그대로(범주형 Embedding + LSTM).

- 구조: `[연속형 + Driver/Compound/Race Embedding] → LSTM(128)×2 → Dropout(0.1) → Linear(1)`
- 검증: **GroupKFold(Race+Year) 5-fold OOF**
- **CSV 비교 로그**: 하이퍼파라미터 조합별 **소요시간 / 연산량(params·throughput·GPU mem) / 결과(AUC·F1)** 를 `lstm_cuda_results.csv`에 기록

> ⚠️ 5060 Ti(Blackwell) → PyTorch cu128(2.7+) & CUDA 12.8+ 필요. 첫 셀에서 `device cuda` 확인.

In [ ]:
import os, gc, time, copy
from itertools import product
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

USE_AMP     = (device.type == 'cuda')
PIN         = (device.type == 'cuda')
NUM_WORKERS = 0

# ===== 실험 설정 =====
WINDOW_SIZE  = 5
EMB_DIMS     = {'Driver': 16, 'Compound': 3, 'Race': 6}
POS_WEIGHT   = 4.03
RESULTS_CSV  = 'lstm_cuda_results.csv'

print('torch', torch.__version__, '| device', device, '| AMP', USE_AMP)
if device.type == 'cuda':
    print('GPU :', torch.cuda.get_device_name(0))

## 1. 데이터 로드 · 정렬 · 인코딩 · 슬라이딩 윈도우 (Step 1·2·4)

In [ ]:
data_path = './data/kaggle_data/'
train = pd.read_csv(data_path + 'train.csv')
test  = pd.read_csv(data_path + 'test.csv')

cont_features = ['LapNumber', 'TyreLife', 'LapTime (s)', 'LapTime_Delta',
                 'Cumulative_Degradation', 'RaceProgress', 'Position', 'Position_Change']
cat_features  = ['Driver', 'Compound', 'Race']
group_keys    = ['Driver', 'Race', 'Year']
target_col    = 'PitNextLap'

# Step1 정렬
train = train.sort_values(['Driver', 'Race', 'Year', 'LapNumber']).reset_index(drop=True)

# Step2 범주형 인코딩 (train+test 합쳐 fit)
cat_dims = []
for col in cat_features:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]], axis=0).astype(str))
    train[col + '_enc'] = le.transform(train[col].astype(str))
    test[col + '_enc']  = le.transform(test[col].astype(str))
    cat_dims.append(len(le.classes_))
cat_enc  = [c + '_enc' for c in cat_features]
emb_dims = [EMB_DIMS[c] for c in cat_features]
print('cat_dims', dict(zip(cat_features, cat_dims)), '| emb_dims', emb_dims)

def make_sliding_windows(df, window_size):
    Xn, Xc, ys, grp = [], [], [], []
    for keys, g in df.groupby(group_keys):
        g = g.sort_values('LapNumber')
        cont = g[cont_features].values; cat = g[cat_enc].values; tgt = g[target_col].values
        ry = f'{keys[1]}_{keys[2]}'
        for i in range(window_size, len(g)):
            Xn.append(cont[i-window_size:i]); Xc.append(cat[i-window_size:i])
            ys.append(tgt[i]); grp.append(ry)
    return (np.array(Xn, dtype='float32'), np.array(Xc, dtype='int64'),
            np.array(ys, dtype='float32'), np.array(grp))

Xn, Xc, y, groups = make_sliding_windows(train, WINDOW_SIZE)
print('Xn', Xn.shape, '| Xc', Xc.shape, '| pos rate', round(float(y.mean()), 4))

## 2. 모델 · Dataset · 학습/평가 유틸 (Step 6·7)

Keras `LSTM(128,return_sequences=True)→LSTM(128)`는 PyTorch `nn.LSTM(num_layers=2)` 후 마지막 timestep으로 대응.
EarlyStopping은 계획서대로 **val_loss / patience=epochs*0.3**.

In [ ]:
class PitLSTM(nn.Module):
    def __init__(self, n_cont, cat_dims, emb_dims, hidden=128, dropout=0.1):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(c, d) for c, d in zip(cat_dims, emb_dims)])
        in_dim = n_cont + sum(emb_dims)
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=2, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, xn, xc):
        embs = [self.embs[i](xc[:, :, i]) for i in range(xc.shape[2])]
        x = torch.cat([xn] + embs, dim=-1)
        out, _ = self.lstm(x)
        out = self.drop(out[:, -1, :])
        return self.fc(out).squeeze(-1)

class WinDataset(Dataset):
    def __init__(self, Xn, Xc, y=None):
        self.Xn = torch.from_numpy(Xn); self.Xc = torch.from_numpy(Xc)
        self.y = torch.from_numpy(y) if y is not None else None
    def __len__(self):
        return len(self.Xn)
    def __getitem__(self, i):
        if self.y is not None:
            return self.Xn[i], self.Xc[i], self.y[i]
        return self.Xn[i], self.Xc[i]

def amp_ctx():
    return torch.autocast(device_type='cuda', enabled=True) if USE_AMP else nullcontext()

def count_params():
    m = PitLSTM(len(cont_features), cat_dims, emb_dims)
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
N_PARAMS = count_params()
print('학습 파라미터 수:', f'{N_PARAMS:,}')

In [ ]:
def eval_model(model, Xn, Xc, yv, crit, bs=8192):
    model.eval(); probs = []; loss_sum = 0.0; n = 0
    loader = DataLoader(WinDataset(Xn, Xc, yv), batch_size=bs, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN)
    with torch.no_grad():
        for xn, xc, yy in loader:
            xn = xn.to(device, non_blocking=True); xc = xc.to(device, non_blocking=True)
            yy = yy.to(device, non_blocking=True)
            with amp_ctx():
                logit = model(xn, xc)
                loss = crit(logit, yy)
            loss_sum += loss.item() * len(yy); n += len(yy)
            probs.append(torch.sigmoid(logit.float()).cpu().numpy())
    return loss_sum / n, np.concatenate(probs)

def train_model(Xn_tr, Xc_tr, y_tr, Xn_va, Xc_va, y_va, epochs, batch_size,
                lr=1e-3, dropout=0.1):
    model = PitLSTM(len(cont_features), cat_dims, emb_dims, dropout=dropout).to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=device))
    opt = optim.Adam(model.parameters(), lr=lr)
    scaler_amp = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    tr_loader = DataLoader(WinDataset(Xn_tr, Xc_tr, y_tr), batch_size=batch_size,
                           shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN)
    patience = max(1, int(epochs * 0.3))
    best_loss, best_state, no_improve, epochs_ran = float('inf'), None, 0, 0
    for ep in range(epochs):
        model.train()
        for xn, xc, yy in tr_loader:
            xn = xn.to(device, non_blocking=True); xc = xc.to(device, non_blocking=True)
            yy = yy.to(device, non_blocking=True)
            opt.zero_grad()
            with amp_ctx():
                loss = crit(model(xn, xc), yy)
            scaler_amp.scale(loss).backward(); scaler_amp.step(opt); scaler_amp.update()
        epochs_ran += 1
        vloss, _ = eval_model(model, Xn_va, Xc_va, y_va, crit)
        if vloss < best_loss - 1e-5:
            best_loss, best_state, no_improve = vloss, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    model.load_state_dict(best_state)
    return model, epochs_ran

## 3. GroupKFold OOF + 연산량/시간 계측 (Step 3·5·9)

fold마다 연속형 스케일링(train fold fit) → 학습 → OOF 예측. 동시에 **시간·epoch·throughput·GPU 메모리**를 수집.

In [ ]:
def run_oof(Xn, Xc, y, groups, epochs, batch_size, n_splits=5, lr=1e-3, dropout=0.1):
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(Xn)); fold_aucs = []
    total_epochs, samples = 0, 0
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=device))
    for tr, va in gkf.split(Xn, y, groups=groups):
        sc = StandardScaler()
        nf = Xn.shape[2]
        Xn_tr = sc.fit_transform(Xn[tr].reshape(-1, nf)).reshape(Xn[tr].shape).astype('float32')
        Xn_va = sc.transform(Xn[va].reshape(-1, nf)).reshape(Xn[va].shape).astype('float32')
        model, ep_ran = train_model(Xn_tr, Xc[tr], y[tr], Xn_va, Xc[va], y[va],
                                    epochs, batch_size, lr, dropout)
        _, p = eval_model(model, Xn_va, Xc[va], y[va], crit)
        oof[va] = p; fold_aucs.append(roc_auc_score(y[va], p))
        total_epochs += ep_ran; samples += ep_ran * len(tr)
        del model; gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e6 if device.type == 'cuda' else float('nan')
    stats = {'train_time_s': round(elapsed, 1), 'epochs_ran': total_epochs,
             'samples_processed': samples,
             'throughput_sps': round(samples / elapsed) if elapsed > 0 else 0,
             'peak_mem_mb': round(peak_mem, 1)}
    return oof, fold_aucs, stats

## 4. HPO grid {epochs, batch_size} — 하이퍼파라미터별 비교 (Step 8)

각 조합을 GroupKFold 2-fold(표본 4만)로 평가하고 **시간·연산량·결과를 CSV 행으로 누적**한다.

In [ ]:
param_grid = {'epochs': [20, 40], 'batch_size': [256, 512]}   # CUDA는 큰 배치가 유리
rows = []

# HPO용 표본 (window=5 고정)
rng = np.random.default_rng(SEED)
samp = rng.choice(len(Xn), size=min(40000, len(Xn)), replace=False)
Xn_s, Xc_s, y_s, g_s = Xn[samp], Xc[samp], y[samp], groups[samp]

for ep, bs in product(param_grid['epochs'], param_grid['batch_size']):
    print(f'[grid] epochs={ep}, batch_size={bs}')
    oof_s, fa, st = run_oof(Xn_s, Xc_s, y_s, g_s, ep, bs, n_splits=2)
    rows.append({'phase': 'hpo', 'model': 'LSTM_emb', 'window_size': 5,
                 'epochs': ep, 'batch_size': bs, 'lr': 1e-3, 'dropout': 0.1,
                 'n_params': N_PARAMS,
                 'oof_auc': round(roc_auc_score(y_s, oof_s), 4),
                 'fold_mean_auc': round(float(np.mean(fa)), 4),
                 'fold_std_auc': round(float(np.std(fa)), 4),
                 'f1': round(f1_score(y_s, (oof_s >= 0.5).astype(int)), 4),
                 **st, 'device': device.type})
    print('   ', st, 'AUC', rows[-1]['oof_auc'])

hpo_df = pd.DataFrame(rows).sort_values('oof_auc', ascending=False).reset_index(drop=True)
best_epochs = int(hpo_df.loc[0, 'epochs']); best_bs = int(hpo_df.loc[0, 'batch_size'])
print('best:', {'epochs': best_epochs, 'batch_size': best_bs})
hpo_df

## 5. 최종 5-fold + window 민감도 — CSV 기록 (Step 9·10)

In [ ]:
def append_row(rows, phase, ws, oof, fa, st, ep, bs):
    rows.append({'phase': phase, 'model': 'LSTM_emb', 'window_size': ws,
                 'epochs': ep, 'batch_size': bs, 'lr': 1e-3, 'dropout': 0.1,
                 'n_params': N_PARAMS,
                 'oof_auc': round(roc_auc_score(oof[1], oof[0]), 4),
                 'fold_mean_auc': round(float(np.mean(fa)), 4),
                 'fold_std_auc': round(float(np.std(fa)), 4),
                 'f1': round(f1_score(oof[1], (oof[0] >= 0.5).astype(int)), 4),
                 **st, 'device': device.type})

# 최종 5-fold (window=5, best params)
oof5, fa5, st5 = run_oof(Xn, Xc, y, groups, best_epochs, best_bs, n_splits=5)
append_row(rows, 'final_5fold', 5, (oof5, y), fa5, st5, best_epochs, best_bs)
print('final 5-fold OOF AUC', rows[-1]['oof_auc'], st5)

# window 민감도 (3 / 5 / 10)
for ws in [3, 5, 10]:
    Xnw, Xcw, yw, gw = make_sliding_windows(train, ws)
    oofw, faw, stw = run_oof(Xnw, Xcw, yw, gw, best_epochs, best_bs, n_splits=5)
    append_row(rows, 'window_sens', ws, (oofw, yw), faw, stw, best_epochs, best_bs)
    print(f'  window={ws}  AUC', rows[-1]['oof_auc'], '| time', stw['train_time_s'], 's')

results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_CSV, index=False)
print('\nsaved', RESULTS_CSV, results_df.shape)
results_df

## 6. 평가 지표 (F1 / Confusion / ROC) (Step 11)

In [ ]:
pred_label = (oof5 >= 0.5).astype(int)
print('OOF AUC :', round(roc_auc_score(y, oof5), 4))
print('F1      :', round(f1_score(y, pred_label), 4))
print('Confusion Matrix:\n', confusion_matrix(y, pred_label))

fpr, tpr, _ = roc_curve(y, oof5)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f'LSTM (AUC={roc_auc_score(y, oof5):.4f})')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('LSTM ROC Curve'); plt.legend(); plt.show()

## 7. 전체 재학습 → test 추론 → 제출 (Step 12)

In [ ]:
def make_predict_windows(df, window_size):
    ids, Xn_list, Xc_list = [], [], []
    for keys, g in df.groupby(group_keys):
        g = g.sort_values('LapNumber')
        cont = g[cont_features].values; cat = g[cat_enc].values; idv = g['id'].values
        for i in range(len(g)):
            wn = cont[max(0, i - window_size):i]; wc = cat[max(0, i - window_size):i]
            if len(wn) < window_size:
                pad_n = np.repeat(cont[0:1], window_size - len(wn), axis=0)
                pad_c = np.repeat(cat[0:1], window_size - len(wc), axis=0)
                wn = np.vstack([pad_n, wn]) if len(wn) > 0 else pad_n
                wc = np.vstack([pad_c, wc]) if len(wc) > 0 else pad_c
            ids.append(idv[i]); Xn_list.append(wn); Xc_list.append(wc)
    return (np.array(ids), np.array(Xn_list, dtype='float32'), np.array(Xc_list, dtype='int64'))

# 전체 train 재학습 (best params), 스케일러는 전체 train fit
sc_all = StandardScaler(); nf = Xn.shape[2]
Xn_all = sc_all.fit_transform(Xn.reshape(-1, nf)).reshape(Xn.shape).astype('float32')
# 간이 hold-out (마지막 그룹 일부)로 EarlyStopping 모니터
gkf = GroupKFold(n_splits=5); tr_i, va_i = next(gkf.split(Xn_all, y, groups=groups))
final, _ = train_model(Xn_all[tr_i], Xc[tr_i], y[tr_i], Xn_all[va_i], Xc[va_i], y[va_i],
                       best_epochs, best_bs)
torch.save(final.state_dict(), 'best_lstm_cuda.pth')

ids_test, Xn_te, Xc_te = make_predict_windows(test, WINDOW_SIZE)
Xn_te = sc_all.transform(Xn_te.reshape(-1, nf)).reshape(Xn_te.shape).astype('float32')
loader = DataLoader(WinDataset(Xn_te, Xc_te), batch_size=8192, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=PIN)
final.eval(); proba = []
with torch.no_grad():
    for xn, xc in loader:
        xn = xn.to(device); xc = xc.to(device)
        with amp_ctx():
            logit = final(xn, xc)
        proba.append(torch.sigmoid(logit.float()).cpu().numpy())
proba = np.concatenate(proba)

sub = pd.DataFrame({'id': ids_test, 'PitNextLap': proba})
sub = sub.set_index('id').loc[test['id']].reset_index()
sub.to_csv('submission_lstm_cuda.csv', index=False)
print('saved submission_lstm_cuda.csv', sub.shape); sub.head()

## 8. 결과 비교 CSV 요약

`lstm_cuda_results.csv` — 하이퍼파라미터 조합별 소요시간 / 연산량 / 결과 비교표.

In [ ]:
df = pd.read_csv(RESULTS_CSV)
cols = ['phase', 'window_size', 'epochs', 'batch_size', 'oof_auc', 'fold_std_auc',
        'f1', 'train_time_s', 'epochs_ran', 'throughput_sps', 'peak_mem_mb']
print(df[cols].to_string(index=False))